In [18]:
# --- Core scientific stack ---
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import csv

# --- Text & NLP ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import spacy

# --- Manifold / embeddings / distance metrics ---
from sklearn.manifold import SpectralEmbedding
from sklearn.metrics import pairwise_distances

In [19]:
import cupy as cp
import numpy as np
from cuml.feature_extraction.text import TfidfVectorizer
from cuml.metrics import pairwise_distances
import cudf as cd
from cudf import Series
import pandas as pd
import csv

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import SpectralEmbedding
from cuml.metrics import pairwise_distances
import spacy

from collections import defaultdict

from pathlib import Path

ModuleNotFoundError: No module named 'cupy'

In [17]:
!pip install cupy
!pip install cuml
!pip install cudf



  Using cached cupy-13.6.0.tar.gz (3.3 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl.metadata (7.7 kB)
Using cached fastrlock-0.8.3-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_28_x86_64.whl (53 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for cupy (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [132 lines of output]
      Clearing directory: /tmp/pip-install-s0cvzzqe/cupy_da6fc400bfbe44cda82c3a792ee34d4f/cupy/.data
      Generating CUPY_CACHE_KEY from header files...
      CUPY_CACHE_KEY (1717 files matching /tmp/pip-install-s0cvzzqe/cupy_da6fc400bfbe44cda82c3a792ee34d4f/cupy/_core/include/**): 3b917dd1428e7d3cdebf5768775abe84f10b0153
      
      -------- Configuring Module: cuda --------
      
      -------- Configuring Module: cu

In [20]:
# load spacy
nlp = spacy.load("en_core_web_lg")

In [21]:
sbert = SentenceTransformer("all-MiniLM-L12-v2")

# Point this to the path containing all goose_<STATE> folders
input_dir = Path('data/raw_text/')

df_files = list()
for dirpath, dirs, files in input_dir.walk():
    for f in files:
        if Path(f).suffix.lower() == '.csv':
            df_files.append(dirpath.joinpath(f))
df = pd.concat([pd.read_csv(f) for f in df_files])
df = df.dropna(subset=['text'])  # Drop NA text

df.head()

ValueError: No objects to concatenate

In [ ]:
# Extract the first paragraph
def get_doc_verbs(doc):
    return " ".join([t.text for t in doc if t.pos_ == "VERB"])
def get_doc_nouns(doc):
    return " ".join([t.text for t in doc if t.pos_ in {'NOUN', 'PROPN'}])

df['1st_para'] = df['text'].str.strip().str.extract(r"(.*)\n")  # This regex will extract the first line (paragraph?) of the document.

docs = nlp.pipe(df['1st_para'].astype(str).to_list())

df['1st_para_verbs'] = pd.Series(get_doc_verbs(doc) for doc in docs)
df['1st_para_nouns'] = pd.Series(get_doc_nouns(doc) for doc in docs)


In [ ]:
x = sbert.encode(list(df['text'].astype(str)))
x_1st_para = sbert.encode(df['1st_para'].astype(str).to_list())
x_noun = sbert.encode(df['1st_para_nouns'].astype(str).to_list())
x_verb = sbert.encode(df['1st_para_verbs'].astype(str).to_list())


In [ ]:
# Get the affinity matrices for each baseline method

a = cosine_similarity(x)  # full SBERT
a_1st_para = cosine_similarity(x_1st_para)
a_noun = cosine_similarity(x_noun)
a_verb = cosine_similarity(x_verb)


# Check that all shapes are identical
assert a.shape == a_1st_para.shape
assert a.shape == a_noun.shape
assert a.shape == a_verb.shape

print("Done.")